In [ ]:
# Notebook 全局导入与超参数。只改这一格，然后从上到下重新运行整个 notebook。
from pathlib import Path
import importlib.util
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
from skopt.space import Categorical, Integer, Real
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

support_path_candidates = [
    Path("cuda_training_support.py"),
    Path("..") / "cuda_training_support.py",
]
SUPPORT_PATH = next((path.resolve() for path in support_path_candidates if path.exists()), None)
if SUPPORT_PATH is None:
    raise FileNotFoundError("找不到 cuda_training_support.py")

support_spec = importlib.util.spec_from_file_location("cuda_training_support", SUPPORT_PATH)
if support_spec is None or support_spec.loader is None:
    raise ImportError(f"无法加载训练辅助模块: {SUPPORT_PATH}")

cuda_training_support = importlib.util.module_from_spec(support_spec)
sys.modules["cuda_training_support"] = cuda_training_support
support_spec.loader.exec_module(cuda_training_support)

PROJECT_ROOT = SUPPORT_PATH.parent

# 如果内核从项目根目录启动，先移除本地 lightgbm 目录对官方包导入的遮蔽。
current_dir = Path.cwd().resolve()
if (current_dir / "lightgbm").is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

import lightgbm as lgb

# NOTEBOOK_RANDOM_SEED: 控制数据切分、BayesSearch、SMOTE 和模型训练的随机种子。
NOTEBOOK_RANDOM_SEED = 114514
# NOTEBOOK_TEST_SIZE: 固定留作最终评估的测试集比例。
NOTEBOOK_TEST_SIZE = 0.25
# NOTEBOOK_BAYES_N_ITER: BayesSearchCV 的候选参数组合采样次数。
NOTEBOOK_BAYES_N_ITER = 24
# NOTEBOOK_CV_FOLDS: 分层交叉验证折数。
NOTEBOOK_CV_FOLDS = 5
# NOTEBOOK_MODEL_N_JOBS: 单个 LightGBM 模型内部使用的线程数。
NOTEBOOK_MODEL_N_JOBS = 1
# NOTEBOOK_SEARCH_N_JOBS: BayesSearchCV 并行 worker 数；GPU 搜索通常保持 1 更稳。
NOTEBOOK_SEARCH_N_JOBS = 1
# NOTEBOOK_SMOTE_K_NEIGHBORS: 每个训练折内部 SMOTE 的近邻数。
NOTEBOOK_SMOTE_K_NEIGHBORS = 5
# NOTEBOOK_BAYES_SCORING: BayesSearchCV 选参使用的目标指标。
NOTEBOOK_BAYES_SCORING = "roc_auc"
# NOTEBOOK_BAYES_VERBOSE: BayesSearchCV 日志级别。
NOTEBOOK_BAYES_VERBOSE = 0
# NOTEBOOK_TQDM_DESC: notebook 中训练进度条的标题。
NOTEBOOK_TQDM_DESC = "LightGBM BayesSearchCV"
NOTEBOOK_MODEL_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_model.txt"
NOTEBOOK_PREPROCESSOR_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_preprocessor.joblib"
NOTEBOOK_MANIFEST_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_inference_assets.json"

NOTEBOOK_LGBM_SEARCH_SPACES = {
    "num_leaves": Integer(8, 31),  # 单棵树的最大叶子数；越小越保守，泛化通常更稳。
    "learning_rate": Real(3e-3, 8e-3, prior="log-uniform"),  # 每轮 boosting 的步长；越小越稳，但通常需要更多树。
    "n_estimators": Integer(500, 1800),  # boosting 轮数；与 learning_rate 联动控制容量。
    "max_depth": Categorical([3, 4, 5, 6]),  # 树深上限；直接限制树结构复杂度。
    "subsample": Real(0.6, 0.85),  # 行采样比例；降低同一批样本反复参与建树的风险。
    "colsample_bytree": Real(0.6, 0.85),  # 列采样比例；降低特征共适应和过拟合。
    "min_child_samples": Integer(50, 200),  # 叶子最少样本数；增大后分裂会更保守。
    "min_split_gain": Real(1e-2, 0.3, prior="log-uniform"),  # 最小分裂增益；越大越不容易继续分裂。
    "reg_alpha": Real(0.1, 10.0, prior="log-uniform"),  # L1 正则强度；鼓励更稀疏的分裂模式。
    "reg_lambda": Real(5.0, 50.0, prior="log-uniform"),  # L2 正则强度；抑制过大的叶子权重。
}

NOTEBOOK_CONFIG = cuda_training_support.build_notebook_run_config(
    bayes_n_iter=NOTEBOOK_BAYES_N_ITER,
    cv_folds=NOTEBOOK_CV_FOLDS,
    random_seed=NOTEBOOK_RANDOM_SEED,
    test_size=NOTEBOOK_TEST_SIZE,
    model_n_jobs=NOTEBOOK_MODEL_N_JOBS,
    search_n_jobs=NOTEBOOK_SEARCH_N_JOBS,
    smote_k_neighbors=NOTEBOOK_SMOTE_K_NEIGHBORS,
)
NOTEBOOK_CV = StratifiedKFold(
    n_splits=NOTEBOOK_CONFIG.cv_folds,
    shuffle=True,
    random_state=NOTEBOOK_CONFIG.random_seed,
)
DEFAULT_TARGET_COLUMN = cuda_training_support.DEFAULT_TARGET_COLUMN
DATA_PATH = cuda_training_support.resolve_data_path(start_dir=Path.cwd())

print(
    cuda_training_support.format_notebook_run_summary(
        NOTEBOOK_CONFIG,
        data_path=DATA_PATH,
    )
)
print(f"模型输出路径: {NOTEBOOK_MODEL_OUTPUT_PATH}")
print(f"预处理输出路径: {NOTEBOOK_PREPROCESSOR_OUTPUT_PATH}")
print(f"推理清单路径: {NOTEBOOK_MANIFEST_OUTPUT_PATH}")



## LightGBM（CUDA-only）

- 第一个代码单元负责 import、路径修正、notebook 全局配置、带注释的超参数定义，以及模型输出路径。
- 第二个代码单元只做数据加载、`RETENTION_TIME` 清洗诊断、训练/测试集切分，不启动训练。
- 第三个代码单元负责唯一一次 `BayesSearchCV` 训练、评估、特征重要性可视化，并输出完整推理资产。
- LightGBM 训练设备固定为 `cuda`，禁止 CPU fallback。
- 缺失值填充、标准化和 SMOTE 都在 estimator 的 `fit` 内部执行；BayesSearchCV 的每个训练折都会单独拟合预处理并单独做 SMOTE，避免数据泄漏。
- `RETENTION_TIME` 会统一解析：普通数值直接保留，多数值文本（如 `17.9 and 18.5`）按均值折叠为单值，真正缺失值保留为缺失，后续再由训练折内众数填补。
- Notebook 训练进度条由 `tqdm.auto` 提供。
- 训练完成后会同时保存 `models/lightgbm_cuda_model.txt`、`models/lightgbm_cuda_preprocessor.joblib` 和 `models/lightgbm_cuda_inference_assets.json`。
- 官方当前不支持 Windows 上的 CUDA 版 LightGBM。需要把训练移动到 Linux 或 WSL2，并先执行：

```bash
pip install -r requirements.txt
pip uninstall -y lightgbm
pip install lightgbm --no-binary lightgbm --config-settings=cmake.define.USE_CUDA=ON
```



In [ ]:
# 加载数据、切分数据并输出训练前诊断。
random_seed = NOTEBOOK_CONFIG.random_seed

data = cuda_training_support.load_training_dataframe(
    data_path=DATA_PATH,
    random_seed=random_seed,
)
retention_time_raw = data["RETENTION_TIME"].copy()

prepared = cuda_training_support.prepare_lightgbm_training_data(
    data,
    target_column=DEFAULT_TARGET_COLUMN,
    random_state=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
)
X_train = prepared["X_train"]
X_test = prepared["X_test"]
y_train = prepared["y_train"]
y_test = prepared["y_test"]
retention_time_diagnostics = prepared["retention_time_diagnostics"]

print("=== 训练前诊断 ===")
print("样本总数:", len(data))
print("标签分布:")
print(data[DEFAULT_TARGET_COLUMN].value_counts())
print()
print("转换前 RETENTION_TIME 的示例值：")
print(retention_time_raw.head())
print()
if retention_time_diagnostics is not None:
    print("RETENTION_TIME 清洗诊断：")
    print(f"严格数值转换后的缺失数: {retention_time_diagnostics['strict_missing_count']}")
    print(f"清洗后的缺失数        : {retention_time_diagnostics['cleaned_missing_count']}")
    print(f"从文本中恢复的记录数  : {retention_time_diagnostics['recovered_from_text_count']}")
    print(f"多值文本记录数        : {retention_time_diagnostics['multi_value_count']}")
    print(f"原始真实缺失数        : {retention_time_diagnostics['original_missing_count']}")
    print(f"仍无法解析的非空记录数: {retention_time_diagnostics['unparsed_non_missing_count']}")
    print("多值文本示例：")
    print(retention_time_diagnostics["multi_value_examples"] or ["<none>"])
    print()
print("训练集形状（原始，SMOTE 将在每个 CV 训练折内部执行）:", X_train.shape)
print("测试集形状:", X_test.shape)
print("训练集标签分布（原始）:")
print(pd.Series(y_train).value_counts())
print()
print("测试集标签分布:")
print(pd.Series(y_test).value_counts())
print()

lgbm_version = cuda_training_support.validate_lightgbm_cuda_build(
    random_state=NOTEBOOK_CONFIG.random_seed,
)
print("LightGBM CUDA preflight:", lgbm_version)



In [ ]:
# 唯一训练块：BayesSearchCV 训练、评估、可视化并保存模型。
lgb_model = cuda_training_support.build_lgbm_classifier(
    random_state=NOTEBOOK_CONFIG.random_seed,
    model_n_jobs=NOTEBOOK_CONFIG.model_n_jobs,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
)

bayes_search = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=NOTEBOOK_LGBM_SEARCH_SPACES,
    n_iter=NOTEBOOK_CONFIG.bayes_n_iter,
    cv=NOTEBOOK_CV,
    scoring=NOTEBOOK_BAYES_SCORING,
    n_jobs=NOTEBOOK_CONFIG.search_n_jobs,
    verbose=NOTEBOOK_BAYES_VERBOSE,
    random_state=NOTEBOOK_CONFIG.random_seed,
)

progress_bar = tqdm(total=NOTEBOOK_CONFIG.bayes_n_iter, desc=NOTEBOOK_TQDM_DESC, unit="iter")
progress_state = {"completed": 0}


def update_training_progress(_optim_result):
    progress_state["completed"] += 1
    progress_bar.update(1)
    progress_bar.set_postfix(completed=progress_state["completed"], refresh=False)
    return False


try:
    bayes_search.fit(X_train, y_train, callback=update_training_progress)
finally:
    progress_bar.close()

best_lgb = bayes_search.best_estimator_

y_train_pred = best_lgb.predict(X_train)
y_train_proba = best_lgb.predict_proba(X_train)[:, 1]
y_test_pred = best_lgb.predict(X_test)
y_test_proba = best_lgb.predict_proba(X_test)[:, 1]

train_metrics = {
    "AUC": roc_auc_score(y_train, y_train_proba),
    "Accuracy": accuracy_score(y_train, y_train_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_train, y_train_pred),
    "Precision": precision_score(y_train, y_train_pred),
    "Recall": recall_score(y_train, y_train_pred),
    "F1": f1_score(y_train, y_train_pred),
}

tn, fp, fn, tp = confusion_matrix(y_train, y_train_pred).ravel()
train_metrics["Specificity"] = tn / (tn + fp)

test_metrics = {
    "AUC": roc_auc_score(y_test, y_test_proba),
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall": recall_score(y_test, y_test_pred),
    "F1": f1_score(y_test, y_test_pred),
}

tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
test_metrics["Specificity"] = tn / (tn + fp)

test_confusion_matrix = confusion_matrix(y_test, y_test_pred)

saved_artifacts = cuda_training_support.save_lightgbm_inference_artifacts(
    estimator=best_lgb,
    prepared=prepared,
    model_path=NOTEBOOK_MODEL_OUTPUT_PATH,
    preprocessor_path=NOTEBOOK_PREPROCESSOR_OUTPUT_PATH,
    manifest_path=NOTEBOOK_MANIFEST_OUTPUT_PATH,
    target_column=DEFAULT_TARGET_COLUMN,
    data_path=DATA_PATH,
    random_seed=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
    scoring=NOTEBOOK_BAYES_SCORING,
)

print("最佳参数组合:", bayes_search.best_params_)
print("最佳交叉验证AUC:", bayes_search.best_score_)
print("最终 refit 前训练集标签分布:", best_lgb.fit_class_counts_)
print("最终 refit 后训练集标签分布（内部 SMOTE）:", best_lgb.resampled_class_counts_)
print(f"模型已保存到: {saved_artifacts['model_path']}")
print(f"预处理包已保存到: {saved_artifacts['preprocessor_path']}")
print(f"推理清单已保存到: {saved_artifacts['manifest_path']}")
print()
print("=== 训练集性能 ===")
for metric, value in train_metrics.items():
    print(f"{metric:<18}: {value:.4f}")
print()
print("=== 测试集性能 ===")
for metric, value in test_metrics.items():
    print(f"{metric:<18}: {value:.4f}")
print()
print("=== 泛化差距（训练集 - 测试集） ===")
for metric in ["AUC", "Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1", "Specificity"]:
    print(f"{metric:<18}: {train_metrics[metric] - test_metrics[metric]:+.4f}")
print()
print("测试集混淆矩阵:")
print(test_confusion_matrix)

plt.figure(figsize=(10, 6))
lgb.plot_importance(best_lgb.booster_, max_num_features=20)
plt.tight_layout()
plt.show()

